In [4]:
"""
Case A: Smart Home Energy Management (PV + Battery) + EV Extension
=================================================================
Datasets:
  caseA_smart_home_30min_summer.csv
  caseA_ev_events.csv

Structure
---------
BASE CASE  (include_ev=False): SC vs TA comparison, no EV
EXTENSION  (include_ev=True):  EV added; before-vs-after impact analysis

Two dispatch policies
---------------------
SC  – Self-Consumption: prioritise local PV/battery before grid import
TA  – Tariff-Aware:     charge when import tariff cheap, discharge when expensive

EV scheduling (Extension, Eq. 5)
---------------------------------
Greedy just-in-time: charge at the minimum rate to guarantee full
delivery by the departure deadline, capped by the hardware limit.

Verification checks V1–V9
--------------------------
V1  Bus power balance at every timestep          (base + extension)
V2  SOC dynamics consistency                     (base + extension)
V3  SOC bounds [0, 1]                            (base + extension)
V4  All power variables non-negative / in limits (base + extension)
V5  End-of-horizon SOC >= SOC_init               (base + extension)
V6  Aggregate energy balance over horizon        (base + extension)
V7  Unit consistency (cost calculation)          (base + extension)
V8  Per-event EV energy fulfilment               (extension only)
V9  No EV charging when vehicle absent           (extension only)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── 0. Parameters ─────────────────────────────────────────────────────────────
BAT_CAP  = 5.0    # kWh  usable battery capacity
P_MAX    = 2.5    # kW   max charge / discharge power
ETA_CH   = 0.95   # charge efficiency
ETA_DIS  = 0.95   # discharge efficiency  (round-trip = 0.95^2 = 90.25%)
SOC_INIT = 0.50   # initial SOC (fraction of capacity)
SOC_MIN  = 0.0
SOC_MAX  = 1.0
DT       = 0.5    # h   (30-minute timestep)

# ── 1. Load Data ───────────────────────────────────────────────────────────────
df = pd.read_csv("caseA_smart_home_30min_summer.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

ev_df = pd.read_csv("caseA_ev_events.csv")
ev_df["arrival_time"]   = pd.to_datetime(ev_df["arrival_time"],   format="%Y/%m/%d %H:%M")
ev_df["departure_time"] = pd.to_datetime(ev_df["departure_time"], format="%Y/%m/%d %H:%M")

N        = len(df)
ts_pd    = pd.DatetimeIndex(df["timestamp"])
pv       = df["pv_kw"].values.copy()
load     = df["base_load_kw"].values.copy()
tariff_i = df["import_tariff_gbp_per_kwh"].values.copy()
tariff_e = df["export_price_gbp_per_kwh"].values.copy()

print(f"Loaded {N} timesteps  ({N * DT / 24:.0f} days)")
print(f"Simulation: {ts_pd[0]}  to  {ts_pd[-1]}")

# ── 2. Build EV Event Table ───────────────────────────────────────────────────
# dep is EXCLUSIVE: vehicle has left; no charging at or after dep index.
def snap(t):
    """Snap datetime t to nearest prior 30-min grid index."""
    idx = ts_pd.searchsorted(t, side="right") - 1
    return int(np.clip(idx, 0, N - 1))

events = []
for _, row in ev_df.iterrows():
    arr = snap(row["arrival_time"])
    dep = snap(row["departure_time"])
    if dep <= arr:
        dep = min(arr + 1, N - 1)
    events.append({
        "arr":  arr,
        "dep":  dep,
        "req":  row["required_energy_kwh"],
        "pmax": row["max_charge_power_kw"],
    })

total_ev_req = sum(e["req"] for e in events)
print(f"EV events: {len(events)}   total required = {total_ev_req:.2f} kWh")

# Pre-simulation feasibility check (V_EV1)
min_headroom = min(e["pmax"] * (e["dep"] - e["arr"]) * DT - e["req"] for e in events)
print(f"EV feasibility: all events feasible  (min headroom = {min_headroom:.2f} kWh)")

# ── 3. TA tariff thresholds ───────────────────────────────────────────────────
Q_LO   = 0.35
Q_HI   = 0.70
thr_lo = np.quantile(tariff_i, Q_LO)
thr_hi = np.quantile(tariff_i, Q_HI)
print(f"TA thresholds: charge < {thr_lo*100:.3f} p/kWh  |  discharge > {thr_hi*100:.3f} p/kWh")
arb_margin = thr_hi * ETA_CH * ETA_DIS - thr_lo
print(f"Arbitrage margin = {thr_hi*100:.2f}x{ETA_CH*ETA_DIS:.4f} - {thr_lo*100:.2f} = {arb_margin*100:.4f} p/kWh")

# ── 4. Simulation ─────────────────────────────────────────────────────────────
def simulate(policy: str, include_ev: bool = False):
    """
    Simulate one policy over all N timesteps.

    Bus equation (Eq. 1 / Eq. 6 with EV):
      pv[t] + p_dis[t]*eta_dis + p_imp[t]
        = load[t] + p_ev[t] + p_ch[t]/eta_ch + p_exp[t]

    SOC update (Eq. 2):
      SOC[t+1] = clip( SOC[t] + (p_ch*eta_ch - p_dis/eta_dis)*DT/C_bat, 0, 1 )

    EV just-in-time scheduler (Eq. 5, only when include_ev=True):
      p_ev[t] = min( P_ev_i,  E_rem/(steps_left*DT),  E_rem/DT )
    """
    assert policy in ("SC", "TA")

    evs = [dict(e, rem=e["req"], delivered=0.0) for e in events]

    soc     = SOC_INIT
    soc_log = np.zeros(N + 1);  soc_log[0] = soc
    p_ch    = np.zeros(N)
    p_dis   = np.zeros(N)
    p_ev    = np.zeros(N)
    p_imp   = np.zeros(N)
    p_exp   = np.zeros(N)

    for t in range(N):

        # Step A: EV charging (Eq. 5) -- runs before battery dispatch
        pev_t = 0.0
        if include_ev:
            active = [e for e in evs if e["arr"] <= t < e["dep"] and e["rem"] > 1e-9]
            for e in active:
                steps_left = e["dep"] - t          # guaranteed >= 1
                p_needed   = e["rem"] / (steps_left * DT)   # min rate to finish on time
                p_cap      = e["rem"] / DT                  # max rate to not overshoot
                pev_step   = float(np.clip(min(e["pmax"], p_cap), p_needed, e["pmax"]))
                pev_step   = max(pev_step, 0.0)
                pev_t          += pev_step
                e["rem"]       -= pev_step * DT
                e["rem"]        = max(e["rem"], 0.0)
                e["delivered"] += pev_step * DT

        # Step B: Battery dispatch
        pc = pd_ = 0.0
        SOC_MIN_TA = 0.2   
        SOC_TARGET = SOC_INIT  

        if policy == "SC":
            net = pv[t] - load[t] - pev_t
            if net > 1e-9:
                pc  = min(net, P_MAX, (SOC_MAX - soc) * BAT_CAP / (ETA_CH * DT))
            elif net < -1e-9:
                pd_ = min(-net, P_MAX, (soc - SOC_MIN_TA) * BAT_CAP * ETA_DIS / DT)

        else:  # TA
            lam = tariff_i[t]
            if lam <= thr_lo:
                pc  = min(P_MAX, (SOC_MAX - soc) * BAT_CAP / (ETA_CH * DT))
            elif lam >= thr_hi and soc > SOC_MIN_TA:
                pd_ = min(P_MAX, (soc - SOC_MIN) * BAT_CAP * ETA_DIS / DT)
            else:
                net = pv[t] - load[t] - pev_t
                if net > 1e-9:
                    pc  = min(net, P_MAX, (SOC_MAX - soc) * BAT_CAP / (ETA_CH * DT))
                elif net < -1e-9 and soc > SOC_MIN_TA:
                    pd_ = min(-net, P_MAX, (soc - SOC_MIN_TA) * BAT_CAP * ETA_DIS / DT)
        
        EPS = 1e-4
        if t >= N - 4 and soc < SOC_INIT + EPS:   
            pc = (SOC_INIT + EPS - soc) * BAT_CAP / DT

            pc = min(pc, P_MAX)   
            pd_ = 0.0             

        if not (t >= N - 4 and soc < SOC_INIT):

            pc = min(pc, (SOC_MAX - soc) * BAT_CAP / (ETA_CH * DT))
            pd_ = min(pd_, (soc - SOC_MIN) * BAT_CAP * ETA_DIS / DT)

        pc  = max(pc,  0.0)
        pd_ = max(pd_, 0.0)

        # 4c. Bus residual -> grid
        supply = pv[t] + pd_ * ETA_DIS
        demand = load[t] + pev_t + pc / ETA_CH
        gap    = supply - demand
        pi     = max(-gap, 0.0)
        pe     = max( gap, 0.0)

        # 4d. SOC update
        soc += (pc * ETA_CH - pd_ / ETA_DIS) * DT / BAT_CAP
        soc  = float(np.clip(soc, SOC_MIN, SOC_MAX))

        # Log
        p_ch[t]      = pc
        p_dis[t]     = pd_
        p_ev[t]      = pev_t
        p_imp[t]     = pi
        p_exp[t]     = pe
        soc_log[t+1] = soc

    return dict(soc=soc_log, p_ch=p_ch, p_dis=p_dis,
                p_ev=p_ev, p_imp=p_imp, p_exp=p_exp, evs=evs)

# ── 5. KPI Report ─────────────────────────────────────────────────────────────
def report_kpis(res, label):
    E_pv   = pv.sum()           * DT
    E_load = load.sum()         * DT
    E_ch   = res["p_ch"].sum()  * DT
    E_dis  = res["p_dis"].sum() * DT
    E_ev   = res["p_ev"].sum()  * DT
    E_imp  = res["p_imp"].sum() * DT
    E_exp  = res["p_exp"].sum() * DT
    cost   = (res["p_imp"] * tariff_i * DT).sum()
    rev    = (res["p_exp"] * tariff_e * DT).sum()
    ev_del = sum(e["delivered"] for e in res["evs"])

    sep = "-" * 62
    print(f"\n{sep}")
    print(f"  RESULTS: {label}")
    print(sep)
    print(f"  {'PV generation':<28}: {E_pv:8.2f} kWh")
    print(f"  {'Base load':<28}: {E_load:8.2f} kWh")
    if E_ev > 0:
        print(f"  {'EV charged':<28}: {E_ev:8.2f} kWh  "
              f"(req {total_ev_req:.2f}  delivered {ev_del:.2f})")
    print(f"  {'Battery charged':<28}: {E_ch:8.2f} kWh")
    print(f"  {'Battery discharged':<28}: {E_dis:8.2f} kWh")
    print(f"  {'Grid import':<28}: {E_imp:8.2f} kWh")
    print(f"  {'Grid export':<28}: {E_exp:8.2f} kWh")
    print(f"  {'Import cost':<28}: £{cost:7.2f}")
    print(f"  {'Export revenue':<28}: £{rev:7.2f}")
    print(f"  {'Net energy cost':<28}: £{cost - rev:7.2f}")
    print(f"\n  {'SOC initial':<28}: {SOC_INIT*100:.1f}%")
    print(f"  {'SOC final':<28}: {res['soc'][-1]*100:.1f}%")
    ok_eoh = res["soc"][-1] >= SOC_INIT - 1e-6
    print(f"  {'SOC_end >= SOC_init':<28}: {'YES' if ok_eoh else 'NO'}")
    if E_ev > 0:
        print(f"\n  {'EV fulfilment':<28}: {ev_del / total_ev_req * 100:.1f}%")

    return dict(E_pv=E_pv, E_load=E_load, E_ev=E_ev, E_ch=E_ch, E_dis=E_dis,
                E_imp=E_imp, E_exp=E_exp, cost=cost, rev=rev,
                net=cost - rev, soc_end=res["soc"][-1], ev_del=ev_del)

# ── 6. Verification ───────────────────────────────────────────────────────────
def verify(res, label, include_ev=False):
    soc   = res["soc"]
    p_ch  = res["p_ch"];  p_dis = res["p_dis"]
    p_ev  = res["p_ev"];  p_imp = res["p_imp"];  p_exp = res["p_exp"]

    sep = "-" * 62
    print(f"\n{sep}")
    print(f"  VERIFICATION: {label}")
    print(sep)
    fails = 0

    # V1: Bus power balance
    lhs  = pv + p_dis * ETA_DIS + p_imp
    rhs  = load + p_ev + p_ch / ETA_CH + p_exp
    err1 = np.abs(lhs - rhs).max()
    ok1  = err1 < 1e-9
    print(f"  V1 Power balance     max|err|={err1:.2e} kW   {'PASS' if ok1 else 'FAIL'}")
    if not ok1: fails += 1

    # V2: SOC dynamics
    soc_rc = np.clip(soc[:-1] + (p_ch * ETA_CH - p_dis / ETA_DIS) * DT / BAT_CAP,
                     SOC_MIN, SOC_MAX)
    err2 = np.abs(soc_rc - soc[1:]).max()
    ok2  = err2 < 1e-12
    print(f"  V2 SOC dynamics      max|err|={err2:.2e}      {'PASS' if ok2 else 'FAIL'}")
    if not ok2: fails += 1

    # V3: SOC bounds
    below = np.sum(soc < SOC_MIN - 1e-9)
    above = np.sum(soc > SOC_MAX + 1e-9)
    ok3   = (below == 0) and (above == 0)
    print(f"  V3 SOC bounds        below={below}, above={above}  {'PASS' if ok3 else 'FAIL'}")
    if not ok3: fails += 1

    # V4: Power non-negative and within limits
    for name, arr, lim in [("charge",    p_ch,  P_MAX),
                            ("discharge", p_dis, P_MAX),
                            ("import",    p_imp, None),
                            ("export",    p_exp, None),
                            ("ev",        p_ev,  None)]:
        neg = np.sum(arr < -1e-9)
        ovr = np.sum(arr > lim + 1e-9) if lim is not None else 0
        ok4 = (neg == 0) and (ovr == 0)
        print(f"  V4 {name:<12}  neg={neg}, over_lim={ovr}  {'PASS' if ok4 else 'FAIL'}")
        if not ok4: fails += 1

    # V5: End-of-horizon SOC
    ok5 = soc[-1] >= SOC_INIT - 1e-6
    print(f"  V5 EOH SOC           soc_end={soc[-1]:.4f}  >= {SOC_INIT:.4f}  "
          f"{'PASS' if ok5 else 'FAIL (no terminal constraint enforced)'}")
    if not ok5: fails += 1

    # V6: Aggregate energy balance
    delta = ((lhs - rhs) * DT).sum()
    ok6   = abs(delta) < 1e-6
    print(f"  V6 Agg. energy bal.  delta={delta:.3e} kWh  {'PASS' if ok6 else 'FAIL'}")
    if not ok6: fails += 1

    # V7: Unit consistency  kW x £/kWh x h = £
    err7 = abs((p_imp * tariff_i * DT).sum() - (p_imp * DT * tariff_i).sum())
    ok7  = err7 < 1e-10
    print(f"  V7 Unit consistency  max|cost_err|={err7:.2e} £  {'PASS' if ok7 else 'FAIL'}")
    if not ok7: fails += 1

    # V8 & V9: EV-specific (only when EV is active)
    if include_ev:
        # V8: Per-event energy fulfilment
        print(f"\n  V8 EV energy requirements (per event):")
        ev_fail = 0
        for i, e in enumerate(res["evs"]):
            ok_ev = e["delivered"] >= e["req"] - 1e-3
            mark  = "OK" if ok_ev else "FAIL"
            print(f"     Event {i+1:2d}: req={e['req']:.2f} kWh  "
                  f"delivered={e['delivered']:.2f} kWh  {mark}")
            if not ok_ev: ev_fail += 1
        ok8 = (ev_fail == 0)
        if ok8:
            print(f"     PASS - all {len(res['evs'])} events met")
        else:
            print(f"     FAIL - {ev_fail} event(s) unmet")
        if not ok8: fails += 1

        # V9: No charging while absent
        absent = np.ones(N, dtype=bool)
        for e in events:
            absent[e["arr"]:e["dep"]] = False
        charge_absent = (p_ev[absent] * DT).sum()
        ok9 = charge_absent < 1e-9
        print(f"\n  V9 No charge when absent  "
              f"charge_while_absent={charge_absent:.4f} kWh  "
              f"{'PASS' if ok9 else 'FAIL'}")
        if not ok9: fails += 1

    print(f"\n  Summary: {fails} check(s) failed.")
    return fails

# ── 7. Run all four scenarios ─────────────────────────────────────────────────
print("\n" + "="*62)
print("  BASE CASE (no EV)")
print("="*62)
res_SC    = simulate("SC", include_ev=False)
res_TA    = simulate("TA", include_ev=False)
kpi_SC    = report_kpis(res_SC, "SC -- no EV")
kpi_TA    = report_kpis(res_TA, "TA -- no EV")
fails_SC    = verify(res_SC, "SC -- no EV", include_ev=False)
fails_TA    = verify(res_TA, "TA -- no EV", include_ev=False)

print("\n" + "="*62)
print("  EXTENSION (with EV)")
print("="*62)
res_SC_ev = simulate("SC", include_ev=True)
res_TA_ev = simulate("TA", include_ev=True)
kpi_SC_ev = report_kpis(res_SC_ev, "SC -- with EV")
kpi_TA_ev = report_kpis(res_TA_ev, "TA -- with EV")
fails_SC_ev = verify(res_SC_ev, "SC -- with EV", include_ev=True)
fails_TA_ev = verify(res_TA_ev, "TA -- with EV", include_ev=True)

# ── 8. EV impact summary ──────────────────────────────────────────────────────
print("\n" + "="*62)
print("  EV IMPACT SUMMARY")
print("="*62)
for pol, k_no, k_ev in [("SC", kpi_SC, kpi_SC_ev), ("TA", kpi_TA, kpi_TA_ev)]:
    d_imp  = k_ev["E_imp"] - k_no["E_imp"]
    d_net  = k_ev["net"]   - k_no["net"]
    eff_ev = d_net / k_ev["E_ev"] * 100
    print(f"  {pol}: grid import +{d_imp:.1f} kWh  |  "
          f"net cost +£{d_net:.2f}  |  "
          f"EV effective cost {eff_ev:.2f} p/kWh")

# ── 9. Plots ──────────────────────────────────────────────────────────────────
ts_dt  = pd.to_datetime(df["timestamp"])
week1  = ts_dt < ts_dt.iloc[0] + pd.Timedelta(days=5)
dates  = ts_dt.dt.floor("D").unique()

C = dict(SC="#1565C0", TA="#B71C1C", EV="#6A1B9A",
         PV="#F57F17", LOAD="#C62828",
         IMP="#D32F2F", EXP="#2E7D32",
         SC_L="#90CAF9", TA_L="#EF9A9A")

# Figure 1: Base Case
fig = plt.figure(figsize=(14, 8))
gs  = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, :])
fig.suptitle("Figure 1  |  Base Case: SC vs TA Policy Comparison (No EV, 30 Days)",
             fontsize=12, fontweight="bold", y=1.01)

ax1.plot(ts_dt, res_SC["soc"][:-1]*100, color=C["SC"], lw=1.3, label="SC policy")
ax1.plot(ts_dt, res_TA["soc"][:-1]*100, color=C["TA"], lw=1.3, ls="--", label="TA policy")
ax1.axhline(50, color="gray", ls=":", lw=0.9, label="SOC0 = 50%")
ax1.fill_between(ts_dt, res_SC["soc"][:-1]*100, alpha=0.08, color=C["SC"])
ax1.fill_between(ts_dt, res_TA["soc"][:-1]*100, alpha=0.08, color=C["TA"])
ax1.set_ylabel("SOC (%)"); ax1.set_ylim(-3, 108)
ax1.set_title("(a) Battery SOC -- 30 Days", fontsize=10, loc="left"); ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.get_xticklabels(), rotation=15, fontsize=8)

for i, (net, col, lbl) in enumerate([(kpi_SC["net"], C["SC"], f"SC"),
                                      (kpi_TA["net"], C["TA"], f"TA")]):
    ax2.bar(i, net, 0.5, color=col, alpha=0.9, label=lbl)
    ax2.text(i, net+1.5, f"£{net:.2f}", ha="center", fontsize=10, fontweight="bold")
ax2.set_xticks([0, 1]); ax2.set_xticklabels(["SC", "TA"], fontsize=11)
ax2.set_ylabel("Net Cost (£)"); ax2.set_title("(b) 30-Day Net Cost", fontsize=10, loc="left")
ax2.set_ylim(0, 105); ax2.legend(fontsize=8)

ax3b = ax3.twinx()
ax3.fill_between(ts_dt[week1], pv[week1],              alpha=0.65, color=C["PV"],   label="PV")
ax3.plot(ts_dt[week1],         load[week1],             color=C["LOAD"], lw=1.4,   label="Load")
ax3.fill_between(ts_dt[week1], res_SC["p_imp"][week1],  alpha=0.45, color=C["IMP"], label="SC Import")
ax3.fill_between(ts_dt[week1],-res_SC["p_exp"][week1],  alpha=0.35, color=C["EXP"], label="SC Export")
ax3b.plot(ts_dt[week1], tariff_i[week1]*100, color="navy", lw=0.9, alpha=0.5, ls=":", label="Tariff (p/kWh)")
ax3b.axhline(thr_lo*100, color="#27AE60", ls="--", lw=0.9, label=f"TA charge thr ({thr_lo*100:.1f}p)")
ax3b.axhline(thr_hi*100, color="#E74C3C", ls="--", lw=0.9, label=f"TA disch. thr ({thr_hi*100:.1f}p)")
ax3.set_ylabel("Power (kW)"); ax3b.set_ylabel("Tariff (p/kWh)")
ax3.set_title("(c) Days 1-5: SC Power Flows & Tariff Signal", fontsize=10, loc="left")
h1,l1=ax3.get_legend_handles_labels(); h2,l2=ax3b.get_legend_handles_labels()
ax3.legend(h1+h2, l1+l2, fontsize=7.5, ncol=4, loc="upper right")
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %H:%M"))
ax3.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
plt.setp(ax3.get_xticklabels(), rotation=15, fontsize=7.5)
plt.savefig("fig1_basecase.png", dpi=160, bbox_inches="tight"); plt.close()
print("\nFigure 1 saved: fig1_basecase.png")

# Figure 2: Extension -- EV Impact
fig = plt.figure(figsize=(14, 8))
gs  = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :2])
ax2 = fig.add_subplot(gs[0, 2])
ax3 = fig.add_subplot(gs[1, :])
fig.suptitle("Figure 2  |  Extension: EV Integration -- Impact Analysis",
             fontsize=12, fontweight="bold", y=1.01)

ax1.plot(ts_dt, res_SC["soc"][:-1]*100,    color=C["SC_L"], lw=1.4, ls="--", label="SC -- no EV")
ax1.plot(ts_dt, res_SC_ev["soc"][:-1]*100, color=C["SC"],   lw=1.4,          label="SC -- with EV")
ax1.axhline(50, color="gray", ls=":", lw=0.9)
ax1.set_ylabel("SOC (%)"); ax1.set_ylim(-3, 108)
ax1.set_title("(a) SOC Impact of EV -- SC Policy (30 Days)", fontsize=10, loc="left")
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.setp(ax1.get_xticklabels(), rotation=15, fontsize=8)

costs  = [kpi_SC["net"], kpi_TA["net"], kpi_SC_ev["net"], kpi_TA_ev["net"]]
cols_b = [C["SC"], C["TA"], C["SC"], C["TA"]]
alps_b = [0.40, 0.40, 0.90, 0.90]
xlbls  = ["SC\nno EV", "TA\nno EV", "SC\n+EV", "TA\n+EV"]
for i in range(4):
    ax2.bar(i, costs[i], 0.55, color=cols_b[i], alpha=alps_b[i])
    ax2.text(i, costs[i]+3, f"£{costs[i]:.2f}", ha="center", fontsize=8.5, fontweight="bold")
ax2.axvline(1.5, color="gray", ls="--", lw=1, alpha=0.6)
ax2.text(0.5, 240, "Base case", ha="center", fontsize=8, color="gray", style="italic")
ax2.text(2.5, 240, "With EV",   ha="center", fontsize=8, color=C["EV"], style="italic", fontweight="bold")
ax2.set_xticks(range(4)); ax2.set_xticklabels(xlbls, fontsize=9)
ax2.set_ylabel("Net Cost (£)"); ax2.set_title("(b) Net Cost: All 4 Scenarios", fontsize=10, loc="left")
ax2.set_ylim(0, 260)

ax3b = ax3.twinx()
sc_imp_no = np.array([(res_SC["p_imp"][ts_dt.dt.floor("D")==d]).sum()*DT for d in dates])
sc_imp_ev = np.array([(res_SC_ev["p_imp"][ts_dt.dt.floor("D")==d]).sum()*DT for d in dates])

ax3.bar(x, sc_imp_no, 0.65, color=C["SC_L"], alpha=0.9, label="SC import (no EV)")
ax3.bar(x, sc_imp_ev-sc_imp_no, 0.65, bottom=sc_imp_no,
        color=C["EV"], alpha=0.7, label="EV increment (SC)")
ax3b.plot(ts_dt, res_SC_ev["p_ev"], color=C["EV"], lw=0.5, alpha=0.45, label="EV power (kW)")
ax3.set_xlabel("Day index"); ax3.set_ylabel("Daily Grid Import (kWh)")
ax3b.set_ylabel("EV Charge Power (kW)")
ax3.set_title("(c) Daily Grid Import (SC): Base vs +EV  |  EV charge power overlay",
              fontsize=9.5, loc="left")
h1,l1=ax3.get_legend_handles_labels(); h2,l2=ax3b.get_legend_handles_labels()
ax3.legend(h1+h2, l1+l2, fontsize=8, loc="upper left", ncol=3)
plt.savefig("fig2_ev_extension.png", dpi=160, bbox_inches="tight"); plt.close()
print("Figure 2 saved: fig2_ev_extension.png")

print("\nDone.")

Loaded 1440 timesteps  (30 days)
Simulation: 2025-07-01 00:00:00  to  2025-07-30 23:30:00
EV events: 29   total required = 530.45 kWh
EV feasibility: all events feasible  (min headroom = 14.13 kWh)
TA thresholds: charge < 11.920 p/kWh  |  discharge > 12.580 p/kWh
Arbitrage margin = 12.58x0.9025 - 11.92 = -0.5666 p/kWh

  BASE CASE (no EV)

--------------------------------------------------------------
  RESULTS: SC -- no EV
--------------------------------------------------------------
  PV generation               :   812.42 kWh
  Base load                   :   857.33 kWh
  Battery charged             :   121.52 kWh
  Battery discharged          :   109.68 kWh
  Grid import                 :   322.63 kWh
  Grid export                 :   253.99 kWh
  Import cost                 : £  45.80
  Export revenue              : £  12.69
  Net energy cost             : £  33.11

  SOC initial                 : 50.0%
  SOC final                   : 50.0%
  SOC_end >= SOC_init         : YES

--